# verify_response.py 工作原理详解

本 notebook 逐步拆解 `scripts/verify_response.py` 的完整工作流程：
- 它在 BioJSON 流水线中处于什么位置？
- 为什么需要验证？核心思路是什么？
- 每个函数具体做了什么？
- 验证 Prompt 的设计逻辑
- 用真实数据演示验证结果
- JSON 提取的正则逻辑（含边界情况分析）

## 1. 总览：verify_response 在流水线中的位置

BioJSON 的完整流水线分两个阶段：

```
阶段1 (md_to_json.py)           阶段2 (verify_response.py)

MD 原文 --LLM提取--> 提取JSON --LLM验证--> 修正后JSON + 验证报告
md/*.md          json/*_nutri_plant.json    json/*_verified.json
                  (可能有幻觉!)              reports/*_verification.json
```

**一句话总结：** `verify_response.py` 是流水线的**第二阶段** -- 对第一阶段提取出的 JSON 做幻觉检测 + 自动修正。

运行方式（通过 `run.sh`）：
```bash
bash scripts/run.sh verify    # 仅验证
bash scripts/run.sh all       # 提取 + 验证
bash scripts/run.sh test      # 测试模式
```

## 2. 为什么需要验证？

LLM 提取信息时存在一个核心问题：**幻觉（Hallucination）**

即使 prompt 要求找不到就填 NA，LLM 仍然可能：
- **编造不存在的数值**：比如论文中没有 EC number，LLM 凭常识补了一个
- **张冠李戴**：把 Discussion 中引用的别人的结果当成本文的发现
- **虚构定量数据**：论文只说显著降低，LLM 编造了具体的 fold-change

### 验证思路

用 LLM 自己来检查 LLM：
1. 把**原文（MD）** + **每个基因的非 NA 字段**一起发给 LLM
2. 让 LLM 逐字段判断：这个值在原文中**能不能找到依据**？
3. 输出三种判定：
   - SUPPORTED -- 原文明确支持
   - UNCERTAIN -- 有相关内容，但不完全确定
   - UNSUPPORTED -- 原文中找不到（可能是幻觉）
4. 自动把 UNSUPPORTED 的字段修正为 NA

## 3. 逐函数拆解

下面按调用顺序，逐一拆解每个关键函数。

| 函数 | 作用 |
|------|------|
| `get_file_pairs()` | 自动配对 md/name.md 和 json/name_nutri_plant.json |
| `extract_non_na_fields()` | 过滤掉 NA/None 字段，只保留需要验证的 |
| `verify_gene_via_api()` | 核心：构造 prompt -> 调 API -> 解析验证结果 |
| `apply_corrections()` | 把 UNSUPPORTED 字段修正为 NA |
| `verify_file()` | 编排单文件的完整验证流程 |
| `print_summary()` | 汇总所有文件的验证统计 |
| `main()` | 入口函数 |

### 3.1 `get_file_pairs()` -- 自动配对文件

配对规则： `md/{name}.md` <-> `json/{name}_nutri_plant.json`

支持**测试模式**（环境变量 `TEST_MODE=1`），此时只处理指定编号的文件。

In [1]:
import os

# 模拟 get_file_pairs 的逻辑
MD_DIR = os.path.join('..', 'md')
JSON_DIR = os.path.join('..', 'json')

md_files = sorted([f for f in os.listdir(MD_DIR) if f.endswith('.md')])
print(f'MD 目录中有 {len(md_files)} 个文件：')
for f in md_files:
    print(f'   {f}')

print(f'\n尝试配对：')
pairs = []
for md_file in md_files:
    stem = os.path.splitext(md_file)[0]
    json_file = f'{stem}_nutri_plant.json'
    json_path = os.path.join(JSON_DIR, json_file)
    if os.path.exists(json_path):
        pairs.append((md_file, json_file, stem))
        print(f'   OK: {md_file} <-> {json_file}')
    else:
        print(f'   SKIP: {md_file} -> 找不到 {json_file}')

print(f'\n共找到 {len(pairs)} 对可验证文件')

MD 目录中有 7 个文件：
   MinerU_markdown_PIIS009286741731499X_2031567088198815744.md
   MinerU_markdown_mmc10_2031567061837611008.md
   MinerU_markdown_mmc3_2031567019886178304.md
   MinerU_markdown_plcell_v31_5_937_2031566954798968832.md
   MinerU_markdown_s41467-024-51114-1_2031567126488616960.md
   MinerU_markdown_s41586-022-04950-4_(1)_2031567254796566528.md
   MinerU_markdown_tieman2017_2031567451551371264.md

尝试配对：
   OK: MinerU_markdown_PIIS009286741731499X_2031567088198815744.md <-> MinerU_markdown_PIIS009286741731499X_2031567088198815744_nutri_plant.json
   SKIP: MinerU_markdown_mmc10_2031567061837611008.md -> 找不到 MinerU_markdown_mmc10_2031567061837611008_nutri_plant.json
   SKIP: MinerU_markdown_mmc3_2031567019886178304.md -> 找不到 MinerU_markdown_mmc3_2031567019886178304_nutri_plant.json
   SKIP: MinerU_markdown_plcell_v31_5_937_2031566954798968832.md -> 找不到 MinerU_markdown_plcell_v31_5_937_2031566954798968832_nutri_plant.json
   SKIP: MinerU_markdown_s41467-024-51114-1_2031567126488

### 3.2 `extract_non_na_fields(gene_dict)` -- 过滤掉 NA 字段

验证只需要检查**有值的字段**。如果一个字段已经是 NA 或 None，就不需要验证了。

这个函数做的事：
1. 跳过 None 值
2. 跳过字符串 "NA"（不区分大小写）
3. 对于列表类型，过滤掉列表中的 "NA" 元素

In [2]:
def extract_non_na_fields(gene_dict):
    """提取基因字典中所有非 NA 的字段"""
    fields = {}
    for key, value in gene_dict.items():
        if value is None:
            continue
        if isinstance(value, str) and value.strip().upper() == "NA":
            continue
        if isinstance(value, list):
            filtered = [v for v in value if not (isinstance(v, str) and v.strip().upper() == "NA")]
            if filtered:
                fields[key] = filtered
        else:
            fields[key] = value
    return fields

# 演示
demo_gene = {
    "Gene_Name": "ZmPSY1",
    "EC_Number": "NA",
    "Species": "Zea mays",
    "Pathway_Name": None,
    "Key_Intermediate_Metabolites_Affected": ["beta-carotene", "NA", "lycopene"],
    "Other_Phenotypic_Effects": ["NA"],
}

result = extract_non_na_fields(demo_gene)
print('原始字段数:', len(demo_gene))
print('需要验证的字段数:', len(result))
print()
for k, v in result.items():
    print(f'  {k}: {v}')
print()
print('被过滤掉的: EC_Number(NA), Pathway_Name(None), Other_Phenotypic_Effects([NA])')

原始字段数: 6
需要验证的字段数: 3

  Gene_Name: ZmPSY1
  Species: Zea mays
  Key_Intermediate_Metabolites_Affected: ['beta-carotene', 'lycopene']

被过滤掉的: EC_Number(NA), Pathway_Name(None), Other_Phenotypic_Effects([NA])


### 3.3 `verify_gene_via_api()` -- 核心验证函数

这是最关键的函数，流程如下：

```
输入: MD 原文 + 一个基因的字段数据
  |
  +- 1. extract_non_na_fields() -> 只保留有值的字段
  |
  +- 2. 构造 user prompt:
  |     "## 论文原文" + 截断的 MD (前 12000 字符)
  |     "## 待验证的基因字段" + JSON 格式的字段
  |
  +- 3. 调用 LLM API (system=验证prompt, user=上面的内容)
  |
  +- 4. 从 LLM 返回中提取 JSON (正则匹配)
  |
  +- 5. json.loads() 解析 -> 返回验证结果
```

**为什么截断到 12000 字符？** 因为 LLM 有 token 限制。原文 + prompt + 输出不能超过模型的上下文窗口。12000 字符约等于 3000-6000 tokens，给输出留下足够空间。

### 3.4 JSON 提取的正则逻辑（重点！含边界情况）

LLM 返回的文本可能有各种格式，代码用了**两层正则**来提取 JSON：

```python
# 第一层：尝试匹配 ```json ... ``` 代码块
code_match = re.search(r"```json\\s*(.*?)\\s*```", raw, re.DOTALL)
if code_match:
    json_str = code_match.group(1)
else:
    # 第二层：直接提取 { ... }
    brace_match = re.search(r"(\\{.*\\})", raw, re.DOTALL)
    json_str = brace_match.group(1) if brace_match else raw

result = json.loads(json_str)
```

下面用代码演示各种情况：

In [ ]:
import re
import json

def extract_json_current(raw):
    """当前 verify_response.py 中的 JSON 提取逻辑（原样复制）"""
    code_match = re.search(r"```json\s*(.*?)\s*```", raw, re.DOTALL)
    if code_match:
        json_str = code_match.group(1)
    else:
        brace_match = re.search(r"(\{.*\})", raw, re.DOTALL)
        json_str = brace_match.group(1) if brace_match else raw
    return json.loads(json_str)

# --- 测试各种 LLM 返回格式 ---
test_cases = [
    ("OK: 标准代码块",
     '```json\n{"Gene_Name": {"verdict": "SUPPORTED", "reason": "found"}}\n```'),
    
    ("OK: 纯 JSON（无代码块）",
     '{"Gene_Name": {"verdict": "SUPPORTED", "reason": "found"}}'),
    
    ("OK: JSON 前有文字",
     'Here is the result:\n{"Gene_Name": {"verdict": "SUPPORTED", "reason": "found"}}'),
    
    ("FAIL: 代码块内非 JSON",
     '```json\nThis is not valid JSON\n```'),
    
    ("FAIL: 纯文本无大括号",
     'Gene_Name is SUPPORTED because it was found in the paper.'),
]

for name, raw in test_cases:
    try:
        result = extract_json_current(raw)
        print(f'{name}')
        print(f'   -> 解析成功，字段数: {len(result)}')
    except json.JSONDecodeError as e:
        print(f'{name}')
        print(f'   -> JSONDecodeError: {str(e)[:60]}')
    except Exception as e:
        print(f'{name}')
        print(f'   -> 其他错误: {e}')
    print()

### 3.4.1 一个微妙的边界情况

如果 LLM 在代码块里写了垃圾文本，但在代码块**外面**有合法的 JSON，
当前代码会**优先使用代码块里的垃圾**，导致解析失败，而忽略外面的合法 JSON。

这是因为 `if code_match:` 分支成功后就不会走 `else` 里的 `{...}` 提取了。

In [ ]:
# 演示这个边界情况
tricky_raw = '```json\nThis is my analysis summary\n```\n\nActual result: {"Gene_Name": {"verdict": "SUPPORTED", "reason": "found"}}'

print('LLM 返回的内容：')
print(tricky_raw)
print('\n' + '='*50)

# 当前代码的行为
code_match = re.search(r"```json\s*(.*?)\s*```", tricky_raw, re.DOTALL)
print(f'\n第一个正则匹配到了: {bool(code_match)}')
if code_match:
    print(f'匹配内容: "{code_match.group(1)}"')
    print('-> 这不是合法 JSON，但代码不会再尝试第二个正则！')
    print('-> json.loads() 会抛 JSONDecodeError')

# 如果直接用 {} 提取呢？
brace_match = re.search(r"(\{.*\})", tricky_raw, re.DOTALL)
if brace_match:
    print(f'\n如果用大括号提取: "{brace_match.group(1)}"')
    try:
        json.loads(brace_match.group(1))
        print('-> 这个是合法 JSON！但当前代码不会走到这里')
    except:
        print('-> 也不是合法 JSON')

### 3.4.2 更健壮的写法（改进建议）

添加 fallback 机制：如果代码块提取失败，再尝试 `{...}` 提取。

In [ ]:
def extract_json_improved(raw):
    """改进版：代码块解析失败时 fallback 到 {} 提取"""
    # 优先尝试代码块
    code_match = re.search(r"```json\s*(.*?)\s*```", raw, re.DOTALL)
    if code_match:
        try:
            return json.loads(code_match.group(1))
        except json.JSONDecodeError:
            pass  # 代码块内容不是合法 JSON，继续尝试
    
    # fallback: 直接提取 {...}
    brace_match = re.search(r"(\{.*\})", raw, re.DOTALL)
    if brace_match:
        return json.loads(brace_match.group(1))
    
    # 最后的尝试：直接解析整个文本
    return json.loads(raw)

# 对比测试
print('对比：当前代码 vs 改进版')
print('='*50)

for name, raw in test_cases:
    try:
        extract_json_current(raw)
        s1 = 'OK'
    except:
        s1 = 'FAIL'
    
    try:
        extract_json_improved(raw)
        s2 = 'OK'
    except:
        s2 = 'FAIL'
    
    diff = ' <-- 差异!' if s1 != s2 else ''
    print(f'{name}')
    print(f'   当前: {s1}  |  改进: {s2}{diff}')

# 边界情况
print(f'\n边界情况（代码块内垃圾 + 外部合法 JSON）：')
try:
    extract_json_current(tricky_raw)
    print(f'   当前: OK')
except:
    print(f'   当前: FAIL')
try:
    extract_json_improved(tricky_raw)
    print(f'   改进: OK <-- fallback 起作用了！')
except:
    print(f'   改进: FAIL')

### 3.4.3 异常处理总结

```python
try:
    response = client.chat.completions.create(...)  # API 调用
    raw = response.choices[0].message.content        # 取文本
    # ... 正则提取 ...
    result = json.loads(json_str)                    # 解析 JSON
    return result

except json.JSONDecodeError as e:   # json.loads() 解析失败
    # 场景: 正则提取到的文本不是合法 JSON
    # 包括: 代码块内非JSON、纯文本无大括号、JSON格式有误
    return {}

except Exception as e:              # 所有其他异常
    # 场景: API网络错误、认证失败、response结构异常等
    return {}
```

关键理解：
- `re.search` 搜不到**不会抛异常**，只是返回 None
- 当两个正则都搜不到时，`json_str = raw`（原始文本）
- 然后 `json.loads(raw)` 必然失败 -> `JSONDecodeError`
- 被 except 捕获后返回空字典 `{}`，验证流程继续

### 3.5 `apply_corrections()` -- 自动修正幻觉字段

拿到验证结果后，这个函数把所有 **UNSUPPORTED** 的字段自动改为 NA。

注意：**UNCERTAIN** 的字段不会被修正，只有 **UNSUPPORTED** 会。
这是一个保守策略——宁可保留不确定的信息，也不轻易删除。

In [ ]:
def apply_corrections(gene_data, verification_result):
    """根据验证结果修正基因数据"""
    corrections = []
    corrected_gene = dict(gene_data)

    for field_name, verdict_info in verification_result.items():
        if not isinstance(verdict_info, dict):
            continue
        verdict = verdict_info.get("verdict", "").upper()
        reason = verdict_info.get("reason", "")

        if verdict == "UNSUPPORTED":
            old_value = corrected_gene.get(field_name)
            corrected_gene[field_name] = "NA"
            corrections.append({
                "field": field_name,
                "old_value": old_value,
                "new_value": "NA",
                "reason": reason,
            })

    return corrected_gene, corrections

# 演示
demo_gene_data = {
    "Gene_Name": "ZmPSY1",
    "EC_Number": "2.5.1.32",
    "Species": "Zea mays",
    "Quantitative_Nutrient_Change": "3.5-fold increase",
}

demo_verification = {
    "Gene_Name": {"verdict": "SUPPORTED", "reason": "Gene name appears in Abstract"},
    "EC_Number": {"verdict": "UNSUPPORTED", "reason": "No EC number mentioned anywhere in the paper"},
    "Species": {"verdict": "SUPPORTED", "reason": "Zea mays stated in Methods"},
    "Quantitative_Nutrient_Change": {"verdict": "UNCERTAIN", "reason": "Paper says significant increase but no fold-change given"},
}

corrected, corrections = apply_corrections(demo_gene_data, demo_verification)

print('修正前 vs 修正后：')
print(f'{"字段":<35} {"修正前":<20} {"修正后":<10} {"判定"}')
print('-' * 85)
for field in demo_gene_data:
    before = str(demo_gene_data[field])[:18]
    after = str(corrected[field])[:18]
    verdict = demo_verification.get(field, {}).get('verdict', '')
    marker = '[FIX]' if before != after else '     '
    print(f'{marker} {field:<30} {before:<20} {after:<10} {verdict}')

print(f'\n共修正 {len(corrections)} 个字段')
for c in corrections:
    print(f'  - {c["field"]}: "{c["old_value"]}" -> "NA"')
    print(f'    原因: {c["reason"]}')

### 3.6 `verify_file()` -- 单文件完整验证流程

这个函数编排了一对文件（MD + JSON）的完整验证逻辑：

```
verify_file(md_path, json_path, stem)
  |
  +- 1. 读取 MD 原文 和 JSON 数据
  |
  +- 2. 解析 JSON 结构，提取 Genes 列表
  |     (兼容两种 JSON 格式)
  |
  +- 3. 逐基因循环:
  |     for gene in genes:
  |       +- verify_gene_via_api() -> 验证结果
  |       +- apply_corrections()   -> 修正数据
  |       +- 统计 SUPPORTED/UNCERTAIN/UNSUPPORTED 数量
  |
  +- 4. 保存修正后的 JSON -> json/{stem}_nutri_plant_verified.json
  |
  +- 5. 保存验证报告 -> reports/{stem}_verification.json
```

**JSON 结构兼容**：代码自动检测两种格式：
- 嵌套格式: `{"CropNutrientMetabolismGeneExtraction": {"Genes": [...]}}`
- 扁平格式: `{"Genes": [...]}`

### 3.7 `main()` 和 `print_summary()` -- 入口和汇总

`main()` 的流程非常直接：
1. 调用 `get_file_pairs()` 找到所有文件对
2. 逐对调用 `verify_file()`
3. 调用 `print_summary()` 打印汇总统计
4. 通过 `TokenTracker` 输出 token 用量报告

## 4. 验证 Prompt 深度解读

验证的质量取决于 `VERIFY_SYSTEM_PROMPT` 的设计。它定义了 5 个验证标准：

| # | 标准 | 检查什么 |
|---|------|----------|
| 1 | Core gene validity | 基因是否在 Results 中被直接实验验证？ |
| 2 | Trait validity | 基因是否与最终营养产物相关（不只是泛化性状）？ |
| 3 | Directionality consistency | 中间代谢物变化方向是否与最终产物一致？ |
| 4 | Evidence alignment | 证据是否来自 Results 的图表，而非 Discussion 推测？ |
| 5 | Hallucination check | 具体数值、基因名、ID 等是否真的出现在论文中？ |

这些标准直接对应 `configs/nutri_plant.txt` 中 Step 3 的验证逻辑。

**输出格式要求**：严格 JSON，每个字段给出 verdict + reason：
```json
{
  "Gene_Name": {"verdict": "SUPPORTED", "reason": "..."},
  "EC_Number": {"verdict": "UNSUPPORTED", "reason": "..."}
}
```

## 5. 真实案例演示

加载一份真实的验证报告，看看实际验证效果。

In [ ]:
import json

# 加载真实验证报告
report_path = '../reports/MinerU_markdown_PIIS009286741731499X_2031567088198815744_verification.json'
with open(report_path, 'r', encoding='utf-8') as f:
    report = json.load(f)

print(f'验证文件: {report["file"]}')
print(f'验证时间: {report["timestamp"]}')
print(f'基因数量: {len(report["genes"])}')
print()

# 总体统计
s = report['summary']
print('=== 总体统计 ===')
print(f'  验证字段数:  {s["total_fields"]}')
print(f'  SUPPORTED:   {s["supported"]}')
print(f'  UNCERTAIN:   {s["uncertain"]}')
print(f'  UNSUPPORTED: {s["unsupported"]}')
print(f'  修正数:      {s["total_corrections"]}')
if s['total_fields'] > 0:
    print(f'  忠实度:      {s["supported"] / s["total_fields"] * 100:.1f}%')

In [ ]:
# 逐基因查看验证结果分布
print('=== 逐基因验证结果 ===')
print()

for gene_info in report['genes']:
    name = gene_info['gene_name']
    stats = gene_info['stats']
    total = stats['supported'] + stats['uncertain'] + stats['unsupported']
    
    print(f'Gene [{gene_info["gene_index"]}] {name}')
    
    # 简易条形图
    if total > 0:
        bar_s = 'S' * stats['supported']
        bar_u = '?' * stats['uncertain']
        bar_n = 'X' * stats['unsupported']
        print(f'  [{bar_s}{bar_u}{bar_n}]')
        print(f'  SUPPORTED={stats["supported"]}  UNCERTAIN={stats["uncertain"]}  UNSUPPORTED={stats["unsupported"]}')
    
    # 显示修正详情
    if gene_info['corrections']:
        print(f'  修正了 {len(gene_info["corrections"])} 个字段:')
        for c in gene_info['corrections']:
            old_val = str(c['old_value'])[:40]
            print(f'    - {c["field"]}: "{old_val}" -> NA')
    else:
        print(f'  无修正')
    print()

In [ ]:
# 查看某个基因的详细验证结果
gene = report['genes'][0]  # 第一个基因
print(f'=== {gene["gene_name"]} 详细验证结果 ===\n')

for field, info in gene['verification'].items():
    verdict = info['verdict']
    reason = info['reason'][:80]
    
    if verdict == 'SUPPORTED':
        tag = '[OK]  '
    elif verdict == 'UNCERTAIN':
        tag = '[??]  '
    else:
        tag = '[FAIL]'
    
    print(f'{tag} {field}')
    print(f'       {reason}')
    print()

## 6. 数据流总结

### 完整数据流

```
输入                          处理                           输出
----                          ----                           ----
md/{name}.md          +--> get_file_pairs()          json/{name}_nutri_plant_verified.json
                      |        |                     reports/{name}_verification.json
json/{name}_nutri_    |    verify_file()              reports/token_usage_verify_{ts}.json
  plant.json ---------+        |
                          for each gene:
                            extract_non_na_fields()
                            verify_gene_via_api()  --API--> LLM
                            apply_corrections()
```

### 关键设计决策

| 决策 | 原因 |
|------|------|
| 逐基因验证（而非整体） | 每个基因的字段独立，逐个验证更精确，也避免单次请求太长 |
| 截断原文到 12000 字符 | 控制 token 用量，避免超过模型上下文窗口 |
| 只修正 UNSUPPORTED | 保守策略，UNCERTAIN 的可能是截断导致的，不轻易删 |
| 两层正则提取 JSON | 兼容 LLM 不同的输出格式习惯 |
| 异常时返回空字典 | 容错设计，单个基因验证失败不影响其他基因 |